# Bài học 09 - Mẫu thiết kế Siêu nhận thức


## Cài đặt

Sổ tay này trình bày mô hình thiết kế Metacognition sử dụng Microsoft Agent Framework.

**Yêu cầu trước:**
- Triển khai Azure OpenAI đã được cấu hình qua biến môi trường
- Đã xác thực Azure CLI (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Metacognition là gì?

Metacognition là **suy nghĩ về suy nghĩ**. Trong bối cảnh các tác nhân AI, nó có nghĩa là xây dựng các tác nhân có thể:

- **Tự phản ánh** về kết quả và quá trình suy luận của chính mình
- **Phát hiện lỗi** và phục hồi một cách linh hoạt thay vì im lặng thất bại
- **Đánh giá** xem phản hồi của họ có đầy đủ và hữu ích hay không
- **Điều chỉnh** chiến lược khi cách tiếp cận ban đầu không hiệu quả (ví dụ, chuyển sang hệ thống dự phòng)

Một tác nhân metacognitive không chỉ trả lời câu hỏi — mà còn giám sát hiệu suất của chính nó và điều chỉnh ngay lập tức.


## Công cụ Chính và Dự phòng

Một mô hình siêu nhận thức phổ biến là **chiến lược dự phòng**. Đại lý thử công cụ chính trước; nếu nó thất bại (ví dụ: lỗi 404), đại lý sẽ nhận ra sự thất bại và chuyển sang công cụ dự phòng một cách minh bạch.

Điều này phản ánh các hệ thống thực tế, nơi các dịch vụ chính có thể không khả dụng và đại lý phải tự chẩn đoán vấn đề trước khi chọn một con đường thay thế.

Dưới đây chúng tôi định nghĩa hai công cụ tra cứu chuyến bay:
- **Chính** — bao phủ Paris, Tokyo, và Barcelona
- **Dự phòng** — bao phủ Berlin, Sydney, và Thành phố New York


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## Tác nhân Tự Phản Chiếu với Khả năng Phục hồi Lỗi

Tác nhân dưới đây được hướng dẫn thử hệ thống bay chính trước, nhận biết các lỗi, và chuyển sang hệ thống dự phòng một cách minh bạch. Sau mỗi phản hồi, nó tự đánh giá ngắn gọn xem có trả lời đầy đủ câu hỏi của người dùng hay không.


In [ ]:
agent = client.as_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## Mẫu Đánh Giá Bản Thân

Một khía cạnh khác của siêu nhận thức là **đánh giá bản thân**: một tác nhân riêng biệt (hoặc cùng một tác nhân trong lần duyệt thứ hai) xem xét phản hồi để đảm bảo tính đầy đủ, chính xác và hữu ích.

Dưới đây chúng ta tạo một tác nhân `ResponseEvaluator` đánh giá các phản hồi của đại lý du lịch trên ba khía cạnh.


In [ ]:
evaluation_agent = client.as_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## Tóm tắt

Trong bài học này, bạn đã học cách xây dựng **tác nhân siêu nhận thức** sử dụng Khung Agent của Microsoft:

- **Tự phản ánh**: Các tác nhân theo dõi quá trình suy luận của chính họ và thông báo minh bạch những gì đã xảy ra.
- **Phục hồi lỗi với các phương án thay thế**: Mẫu công cụ chính + dự phòng trong đó tác nhân phát hiện sự cố (ví dụ: lỗi 404) và tự động thử nguồn thay thế.
- **Tự đánh giá**: Một tác nhân đánh giá riêng biệt chấm điểm các phản hồi về độ đầy đủ, chính xác và hữu ích.

Những mẫu này làm cho các tác nhân trở nên vững chắc hơn, minh bạch hơn và đáng tin cậy hơn — những đặc điểm quan trọng cho việc triển khai thực tế.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Tuyên bố miễn trừ trách nhiệm**:
Tài liệu này đã được dịch bằng dịch vụ dịch thuật AI [Co-op Translator](https://github.com/Azure/co-op-translator). Mặc dù chúng tôi cố gắng đảm bảo độ chính xác, xin lưu ý rằng bản dịch tự động có thể chứa lỗi hoặc sai sót. Tài liệu gốc bằng ngôn ngữ gốc nên được coi là nguồn tin chính thức. Đối với thông tin quan trọng, nên sử dụng dịch vụ dịch thuật chuyên nghiệp bởi con người. Chúng tôi không chịu trách nhiệm về bất kỳ hiểu lầm hoặc giải thích sai nào phát sinh từ việc sử dụng bản dịch này.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
